In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

In [2]:
df = pd.read_csv('final_exercise_dataset.csv')

In [3]:
df.head()

,bodyPart,equipment,id,name,target,secondaryMuscles,instructions,description,difficulty,category,message
0,waist,body weight,2.0,45° side bend,abs,['obliques'],['Stand with your feet shoulder-width apart an...,The 45° side bend is a bodyweight exercise tar...,Beginner,strength,NaN
1,waist,body weight,3.0,air bike,abs,['hip flexors'],['Lie flat on your back with your hands placed...,The air bike is a bodyweight exercise targetin...,Beginner,strength,NaN
2,waist,body weight,6.0,alternate heel touchers,abs,['obliques'],['Lie flat on your back with your knees bent a...,Alternate heel touchers is a bodyweight exerci...,Beginner,strength,NaN
3,back,cable,7.0,alternate lateral pulldown,lats,"['biceps', 'rhomboids']",['Sit on the cable machine with your back stra...,The alternate lateral pulldown is a cable mach...,Advanced,strength,NaN
4,chest,leverage machine,9.0,assisted chest dip (kneeling),pectorals,"['triceps', 'shoulders']",['Adjust the machine to your desired height an...,The assisted chest dip (kneeling) is a chest-f...,Advanced,strength,NaN


In [4]:
df.isnull().sum()

bodyPart              2
equipment             2
id                    2
name                  2
target                2
secondaryMuscles      2
instructions          2
description           2
difficulty            0
category              2
message             688
dtype: int64

In [5]:
 df.duplicated().sum()

np.int64(0)

In [6]:
df.describe()

,id
count,688.000000
mean,414.087209
std,255.478085
min,2.000000
25%,198.750000
50%,391.500000
75%,610.000000
max,976.000000


In [7]:
df = df.drop(688)
df = df.drop(689)

In [8]:
df = df.drop('message', axis=1)

In [9]:
df.isnull().sum()

bodyPart            0
equipment           0
id                  0
name                0
target              0
secondaryMuscles    0
instructions        0
description         0
difficulty          0
category            0
dtype: int64

In [10]:
print(df['secondaryMuscles'][9])

['obliques', 'lower back']


In [11]:
type(df['secondaryMuscles'][0])

str

In [12]:
import ast
df['secondaryMuscles'] = df['secondaryMuscles'].fillna('[]')
df['secondaryMuscles'] = df['secondaryMuscles'].apply(ast.literal_eval)
df['secondaryMuscles'] = df['secondaryMuscles'].apply(', '.join)

In [13]:
df['instructions'] = df['instructions'].fillna('[]')
df['instructions'] = df['instructions'].apply(ast.literal_eval)
df['instructions'] = df['instructions'].apply(', '.join)

In [14]:
df.head()

,bodyPart,equipment,id,name,target,secondaryMuscles,instructions,description,difficulty,category
0,waist,body weight,2.0,45° side bend,abs,obliques,Stand with your feet shoulder-width apart and ...,The 45° side bend is a bodyweight exercise tar...,Beginner,strength
1,waist,body weight,3.0,air bike,abs,hip flexors,Lie flat on your back with your hands placed b...,The air bike is a bodyweight exercise targetin...,Beginner,strength
2,waist,body weight,6.0,alternate heel touchers,abs,obliques,Lie flat on your back with your knees bent and...,Alternate heel touchers is a bodyweight exerci...,Beginner,strength
3,back,cable,7.0,alternate lateral pulldown,lats,"biceps, rhomboids",Sit on the cable machine with your back straig...,The alternate lateral pulldown is a cable mach...,Advanced,strength
4,chest,leverage machine,9.0,assisted chest dip (kneeling),pectorals,"triceps, shoulders",Adjust the machine to your desired height and ...,The assisted chest dip (kneeling) is a chest-f...,Advanced,strength


In [15]:
x = df[['bodyPart', 'equipment', 'target', 'secondaryMuscles','category']]
y = df['difficulty']

In [16]:
le = LabelEncoder()
y = le.fit_transform(y)

In [17]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [18]:
col = ['bodyPart', 'equipment', 'target', 'secondaryMuscles', 'category']

In [19]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoding', OneHotEncoder(handle_unknown = 'ignore'))
])

In [20]:
preprocesser = ColumnTransformer([
    ('cat', cat_pipeline, col)
])

In [21]:
pipeline = Pipeline([
    ('preprocesser', preprocesser),
    ('model', LogisticRegression(max_iter = 1000))
])

In [22]:
pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocesser',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['bodyPart', 'equipment',
                                                   'target', 'secondaryMuscles',
                                                   'category'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [23]:
y_pred = pipeline.predict(x_test)

In [24]:
accuracy_score(y_test, y_pred)

1.0

In [25]:
joblib.dump(le, 'label_encoder.pkl')

['label_encoder.pkl']

In [26]:
joblib.dump(pipeline, 'exercise_model.pkl')

['exercise_model.pkl']